In [ ]:
##v%matplotlib inline
import sys
import yaml
from IPython.display import Code
import pcse
from datetime import datetime, timedelta
import pprint
import pandas as pd
import numpy as np
import matplotlib
#matplotlib.style.use("ggplot")
import matplotlib.pyplot as plt
#from weather_plotting import plot_weather_variable, plot_weather_variable_cum_sum, plot_gdd_cumsum
#from soil_plotting import SMprofileplot, f_SMcurves, SMdynamicplot
print("This notebook was built with:")
print(f"python version: {sys.version}")
print(f"PCSE version: {pcse.__version__}")
import pcse.models
from pcse.input import WOFOST81SiteDataProvider_SNOMIN
from pcse.base import ParameterProvider
from pcse.input import YAMLCropDataProvider
from pcse.input import WOFOST81SiteDataProvider_Classic #to run PP
from pcse.input import CSVWeatherDataProvider
import os
import xarray as xr
import pickle

This notebook was built with:
python version: 3.10.16 (main, Dec  3 2024, 17:27:57) [Clang 16.0.0 (clang-1600.0.26.4)]
PCSE version: 6.0.9


In [50]:
model_name = "WOFOST81_WLP_MLWB"  #"WOFOST81_NWLP_MLWB_SNOMIN"  #"WOFOST81_WLP_MLWB"  #WOFOST81_NWLP_MLWB_SNOMIN
fixed_sowing_date = True #True  #if True, sowing date is always May the 15th - else, variable sowing dates:
# if adjusted by farmers, sowing date estimated as: 
# The sowing date of the crop was calculated following the method proposed by 
# Carter and Saarikko (1996) for wheat, 
# where the sowing date is specified as the day when the smoothed daily mean temperature 
# exceeds a defined threshold in the spring.
# 7-day moving average of daily mean temperature observations exceeding a threshold temperature of 11 °C
# The sowing date was specified as the day when the middle day of the averaging period crosses the threshold temperature.
# Day of year (DOY) 60 is defined as an earliest start for the calculations.

#co2_fixed = 422.0  #fixed CO2 concentration for all years, 2024 value
co2_fixed = 498.0  #fixed CO2 concentration for all years, 2024 value
#co2_fixed = 572.0
#co2_fixed = 600.0

In [51]:
############# weather data #############
dir_path = "data/met2015Sensivity/lagged"
# List all .csv files in the folder
csv_files = [f for f in os.listdir(dir_path) if f.endswith(".csv")]
weatherfile_name = csv_files[1]
weatherfile = f'{dir_path}/{weatherfile_name}'
# # weather data diagnostic figures
df_weather = pd.read_csv(weatherfile, skiprows=9)
df_weather['TMEAN'] = (df_weather['TMIN'] + df_weather['TMAX']) / 2
df_weather['DAY'] = pd.to_datetime(df_weather['DAY'], format='%Y%m%d')
df_weather['YEAR'] = df_weather['DAY'].dt.year

# years = range(1991, 2021)  # Define the range of years, 2021 is not included
# sowingdates = []
# for year in years:
#     df_weather_year =df_weather.copy()
#     df_weather_year = df_weather_year.loc[df_weather['YEAR']==year]
#     df_weather_year = df_weather_year.iloc[60:]  #select rows from day 60 to end of year
#     df_weather_year['TMEAN_MA7'] = df_weather_year['TMEAN'].rolling(window=7, center=True).mean() #check that it is set in the center of the 7 days
#     first_date_above_11C = df_weather_year[df_weather_year['TMEAN_MA7'] > 11]['DAY'].min()
#     #print(first_date_above_11C)
#     sowingdates.append(first_date_above_11C.strftime("%Y-%m-%d"))

In [52]:
#Excel with data: 
excelfilename = 'data/AIStewardWofostData.xlsx'
loc = 'Jokioinen'  #location
soiltype = 'FIN_CLAY_1'  #soil type
fertamount = 100 #500  #fertilizer amount in kg/ha

############# weather data #############
dir_path = "data/met2015Sensivity/lagged"
# List all .csv files in the folder
csv_files = [f for f in os.listdir(dir_path) if f.endswith(".csv")]

############# sowing dates #############
# if not adjusted by farmers, sowing date always on May the 1st
if fixed_sowing_date:
    years = range(1991, 2021)  # Define the range of years, 2021 is not included
    sowingdates = [] # Generate all dates from May 1 to June 10 for each year
    for year in years:
        #only generate dates for May 15
        sowing_date = datetime(year, 5, 15)
        sowingdates.append(sowing_date.strftime("%Y-%m-%d"))

############# crop ###########
crop_dir = "data/crop" #in the folder crop there is one file called crops.yaml that has a list of crops available in the same folder e.g. springwheat.yaml
crop_dict = YAMLCropDataProvider(fpath=crop_dir)
cname = "springwheat_calibrated" 
#cname = "springwheat_calibrated_adapted" 
#cname = "springwheat_calibrated_adapted2" 
#cname = "springwheat_calibrated_adapted4" 
#cname = "springwheat_calibrated_FlevolandWUR"
#cname = "springwheat_calibrated_NossenIRS1"
vname = "Spring_wheat_101" 
crop_dict.set_active_crop(cname, vname)# activate the crop and variety is necessary 

#co2
#co2_serie = pd.read_excel(excelfilename, sheet_name='CO2', engine='openpyxl', index_col = 'year')


# Initial conditions
initial_conditions = pd.read_excel(excelfilename, sheet_name='initial_conditions', engine='openpyxl', header = 2)
initial_conditions_layers = pd.read_excel(excelfilename, sheet_name='initial_condition_layers', engine='openpyxl', header = 2)
soil_curves = pd.read_excel(excelfilename, sheet_name='Soil_curves', engine='openpyxl', header = 2)
soil_curves.index = soil_curves['SOIL_ID']
wofost_dictionary = pd.read_excel(excelfilename, sheet_name='WOFOSTDictionary', engine='openpyxl')

def import_initial_conditions(soiltype_name, initial_conditions_pd, wofost_dictionary_pd, variable):
    """
    Import initial conditions for a specific soil type from the initial_conditions DataFrame.
    
    Parameters:
    soiltype_name (str): The name of the soil type to filter by.
    initial_conditions_pd (DataFrame): The DataFrame containing initial conditions.
    wofost_dictionary_pd (DataFrame): The DataFrame containing WOFOST dictionary for conversion factors.
    variable (str): The variable to retrieve from the initial conditions.
    
    Returns:
    float: The value of the specified variable for the given soil type, converted to wofost units
    """
    variable_value = initial_conditions_pd[initial_conditions_pd['EID'] == soiltype][variable]
    conversion_factor = wofost_dictionary_pd[wofost_dictionary_pd['This table'] == variable]['Conversion'].values[0] 
    return list(variable_value * conversion_factor)

######### set site (if co2 is fixed):
co2 = co2_fixed #fix CO2

sited = WOFOST81SiteDataProvider_SNOMIN(A0SOM = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'A0SOM')[0],
                                CNRatioBio = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'CNRatioBio')[0],
                                CO2 = co2,
                                FASDIS = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'FASDIS')[0],
                                IFUNRN = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'IFUNRN')[0],
                                KDENIT_REF = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'KDENIT_REF')[0],
                                KNIT_REF = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'KNIT_REF')[0],
                                KSORP = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'KSORP')[0],
                                MRCDIS = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'MRCDIS')[0],
                                NH4ConcR = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'NH4ConcR')[0],
                                NH4I= import_initial_conditions(soiltype, initial_conditions_layers, wofost_dictionary, 'NH4I'),
                                NO3ConcR = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'NO3ConcR')[0],
                                NO3I = import_initial_conditions(soiltype, initial_conditions_layers, wofost_dictionary, 'NO3'),
                                NOTINF= import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'NOTINF')[0], #Maximum fraction of rain not-infiltrating into the soil [0-1], default 0. // Allard 1.0,
                                WFPS_CRIT = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'WFPS_CRIT')[0],
                                SMLIM = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'SMLIM')[0], 
                                SSI = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'SSI')[0],
                                SSMAX = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'SSMAX')[0],
                                WAV = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'ICSW')[0]) 
        
############ soil ############
with open(f'data/soil/soil_{soiltype}.yaml', 'r') as file:
    soil = yaml.safe_load(file)

############### run simulations #############
all_results = []
for weatherfile_name in csv_files:
    weatherfile = f'{dir_path}/{weatherfile_name}'
    weatherdataprovider = CSVWeatherDataProvider(weatherfile)

    outputs = {}
    i = 0

    # if adjusted by farmers, sowing date estimated as: 
    if fixed_sowing_date == False:
        df_weather = pd.read_csv(weatherfile, skiprows=9)
        df_weather['TMEAN'] = (df_weather['TMIN'] + df_weather['TMAX']) / 2
        df_weather['DAY'] = pd.to_datetime(df_weather['DAY'], format='%Y%m%d')
        df_weather['YEAR'] = df_weather['DAY'].dt.year

        years = range(1991, 2021)  # Define the range of years, 2021 is not included
        sowingdates = []
        for year in years:
            df_weather_year =df_weather.copy()
            df_weather_year = df_weather_year.loc[df_weather['YEAR']==year]
            df_weather_year = df_weather_year.iloc[60:]  #select rows from day 60 to end of year
            df_weather_year['TMEAN_MA7'] = df_weather_year['TMEAN'].rolling(window=7, center=True).mean() #check that it is set in the center of the 7 days
            first_date_above_11C = df_weather_year[df_weather_year['TMEAN_MA7'] > 8]['DAY'].min()
            sowingdates.append(first_date_above_11C.strftime("%Y-%m-%d"))

    for sowingdate in sowingdates:
        sowingyear = int(sowingdate[0:4])
        startdate = f"{sowingdate[0:4]}-03-01" #always first of april of the year of sowing
        enddate = f"{sowingdate[0:4]}-11-15" # always 15th of November of the year of sowing
    
        print(f"{cname} at: location {loc} ; soil type {soiltype} ; sowing date {sowingdate} ; fertilizer amount {fertamount} kg/ha, with weather {weatherfile}")
        
        ########### site - in the loop because co2 changes each year ############
        #if co2 variable: get co2 for the year of sowing
        #co2 = co2_serie.loc[int(sowingdate[0:4])]['mean']

        # sited = WOFOST81SiteDataProvider_SNOMIN(A0SOM = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'A0SOM')[0],
        #                         CNRatioBio = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'CNRatioBio')[0],
        #                         CO2 = co2,
        #                         FASDIS = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'FASDIS')[0],
        #                         IFUNRN = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'IFUNRN')[0],
        #                         KDENIT_REF = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'KDENIT_REF')[0],
        #                         KNIT_REF = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'KNIT_REF')[0],
        #                         KSORP = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'KSORP')[0],
        #                         MRCDIS = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'MRCDIS')[0],
        #                         NH4ConcR = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'NH4ConcR')[0],
        #                         NH4I= import_initial_conditions(soiltype, initial_conditions_layers, wofost_dictionary, 'NH4I'),
        #                         NO3ConcR = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'NO3ConcR')[0],
        #                         NO3I = import_initial_conditions(soiltype, initial_conditions_layers, wofost_dictionary, 'NO3'),
        #                         NOTINF= import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'NOTINF')[0], #Maximum fraction of rain not-infiltrating into the soil [0-1], default 0. // Allard 1.0,
        #                         WFPS_CRIT = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'WFPS_CRIT')[0],
        #                         SMLIM = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'SMLIM')[0], 
        #                         SSI = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'SSI')[0],
        #                         SSMAX = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'SSMAX')[0],
        #                         WAV = import_initial_conditions(soiltype, initial_conditions, wofost_dictionary, 'ICSW')[0]) 
        
        ############ parameters provider ############
        # crop, soil, and site data as parameters to the simulation
        parameters = ParameterProvider(cropdata=crop_dict , soildata=soil, sitedata=sited)
        
        ############ management #############
        yaml_agro = f"""
        AgroManagement:
        - {startdate}:
            CropCalendar:
                crop_name: {cname}
                variety_name: {vname}
                crop_start_date: {sowingdate}
                crop_start_type: sowing
                crop_end_date: 
                crop_end_type: maturity
                max_duration: 400
            TimedEvents:
            -   event_signal: apply_n_snomin
                name: Timed N application
                comment: Sowing date is the same as fertilisation date. All fertilizer amounts in kg/ha. Combi-drilling used and thus fertilisers appiled at drilling.
                events_table:
                - {sowingdate}:
                    amount: {fertamount}
                    application_depth: 10
                    cnratio: 0
                    f_orgmat: 0.0
                    f_NH4N: 0.5
                    f_NO3N: 0.5
                    initial_age: 0
            StateEvents: null
        """
        agromanagement = yaml.safe_load(yaml_agro)

        ############### simulation #############
        
        # Wofost model with SNOMIN soil module
        #nitrogen limited Wofost model with SNOMIN soil module; check: https://pcse.readthedocs.io/en/stable/available_models.html
        #wofost = pcse.models.Wofost81_NWLP_MLWB_SNOMIN(parameters, weatherdataprovider, agromanagement)#, output_vars=output_var_additions)
        
        # water limited wofost model
        wofost = pcse.models.Wofost81_WLP_MLWB(parameters, weatherdataprovider, agromanagement)
        wofost.run_till_terminate()
        one_output_df = pd.DataFrame(wofost.get_output())

        ############### set a limit for the harvest date ############
        df_weather = pd.read_csv(weatherfile, skiprows=9)
        df_weather['DAY'] = pd.to_datetime(df_weather['DAY'], format='%Y%m%d')
        df_weather['YEAR'] = df_weather['DAY'].dt.year
        df_weather_year = df_weather.copy()
        df_weather_year = df_weather_year.loc[df_weather['YEAR']==sowingyear]
        df_weather_year = df_weather_year.iloc[212:]  #select rows from day 212to end of year
        df_weather_year['TMIN_MA7'] = df_weather_year['TMIN'].rolling(window=7, center=True).mean() #check that it is set in the center of the 7 days
        cut_off_date = df_weather_year[df_weather_year['TMIN_MA7'] < 3]['DAY'].min()
        cut_off_date = pd.Timestamp(cut_off_date)
        
        #check if the last date of one_output_df is after cut_off_date
        one_output_df['day'] = pd.to_datetime(one_output_df['day'])
        last_date_output = one_output_df['day'].iloc[-1]
        last_date_output = pd.Timestamp(last_date_output)
        ######
        if last_date_output > cut_off_date:
            #replace all the values of WSO by 0 for all the dates before cut_off_date
            one_output_df.loc[one_output_df['day'] <= cut_off_date, 'WSO'] = 0.0
            #delete all the rows after cut_off_date
            one_output_df = one_output_df[one_output_df['day'] <= cut_off_date]
        ######

        outputs[i] = one_output_df

        i = i + 1

    #get steps values from weatherfile name
    parts = weatherfile.split("_")
    # Find positions
    tstep_index = parts.index("Tstep") + 1
    ppstep_index = parts.index("PPstep") + 1

    Tstep = float(parts[tstep_index])
    PPstep = float((parts[ppstep_index]).replace(".csv", ""))
    outputs_df = pd.concat(outputs.values(), ignore_index=True)

    ########
    all_results.append({
        'Tstep': Tstep,
        'PPstep': PPstep,
        'outputs_df': outputs_df
    })

springwheat_calibrated at: location Jokioinen ; soil type FIN_CLAY_1 ; sowing date 1991-05-15 ; fertilizer amount 100 kg/ha, with weather data/met2015Sensivity/lagged/era5_Jokioinen_19900101_20241231_Tstep_4.5_PPstep_5.csv
springwheat_calibrated at: location Jokioinen ; soil type FIN_CLAY_1 ; sowing date 1992-05-15 ; fertilizer amount 100 kg/ha, with weather data/met2015Sensivity/lagged/era5_Jokioinen_19900101_20241231_Tstep_4.5_PPstep_5.csv
springwheat_calibrated at: location Jokioinen ; soil type FIN_CLAY_1 ; sowing date 1993-05-15 ; fertilizer amount 100 kg/ha, with weather data/met2015Sensivity/lagged/era5_Jokioinen_19900101_20241231_Tstep_4.5_PPstep_5.csv
springwheat_calibrated at: location Jokioinen ; soil type FIN_CLAY_1 ; sowing date 1994-05-15 ; fertilizer amount 100 kg/ha, with weather data/met2015Sensivity/lagged/era5_Jokioinen_19900101_20241231_Tstep_4.5_PPstep_5.csv
springwheat_calibrated at: location Jokioinen ; soil type FIN_CLAY_1 ; sowing date 1995-05-15 ; fertilizer a

In [53]:
# Suppose all_results is your list of dicts with DataFrames
with open(f"all_results_Soil{soiltype}_CO2{co2_fixed}_FixedSowingDate{fixed_sowing_date}_Fertilization{fertamount}_CropFile{cname}_{model_name}.pkl", "wb") as f:
    pickle.dump(all_results, f)

In [54]:
all_results

[{'Tstep': 4.5,
  'PPstep': 5.0,
  'outputs_df':             day       DVS       LAI          TAGP          WSO         WLV  \
  0    1991-03-01       NaN       NaN           NaN          NaN         NaN   
  1    1991-03-02       NaN       NaN           NaN          NaN         NaN   
  2    1991-03-03       NaN       NaN           NaN          NaN         NaN   
  3    1991-03-04       NaN       NaN           NaN          NaN         NaN   
  4    1991-03-05       NaN       NaN           NaN          NaN         NaN   
  ...         ...       ...       ...           ...          ...         ...   
  4890 2020-08-05  1.895567  1.737764  10993.742741  5849.159006  660.804563   
  4891 2020-08-06  1.919824  1.457494  11082.088257  5957.473719  559.321953   
  4892 2020-08-07  1.945718  1.092330  11115.896910  6012.242704  424.671421   
  4893 2020-08-08  1.974370  0.777748  11151.607079  6069.034405  306.581448   
  4894 2020-08-09  2.000000  0.540960  11176.713211  6115.422916  215.929